In [1]:
import sys
import os
import slurm_utils

HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
print(eid)
if NULL_TYPE == 'impostor-session':
    IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA), 
                             '_'.join(['impostors',''])+'.pkl')
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = None
filenames = fit_eid(eid,sessdf,impostordict = impostordict)
print('saving to files:',filenames)
"""

ADD_TO_SAVING_PATH = 'all220522'
SUBDIRECTORY = dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA)
SLURM_DIRECTORY = os.path.join(GROUP_HOME,'bensonb','international-brain-lab/prior-localization/slurm/')
OUTPUT_PATH = os.path.join(GROUP_HOME,'bensonb/international-brain-lab/prior-localization/decoding/')
print(SUBDIRECTORY)

IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             SUBDIRECTORY, 
                             '_'.join(['impostors',''])+'.pkl')
if os.path.exists(IMPOSTOR_PATH):
    print('Impostor sessions already exist.')

decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522


In [2]:
# make python file
py_file = os.path.join(OUTPUT_PATH, SUBDIRECTORY, '_'.join(['slurm_decode_eid',''])+'.py')
if not os.path.exists(os.path.join(OUTPUT_PATH, SUBDIRECTORY)):
    os.mkdir(os.path.join(OUTPUT_PATH, SUBDIRECTORY))
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)
    
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')

# sessdf_eids=np.unique(sessdf_eids)[30:35]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']

# collect all target vectors for impostor sessions
if (not os.path.exists(IMPOSTOR_PATH)) and ('impostor-session' in SUBDIRECTORY):
    fimpostor = open(IMPOSTOR_PATH, 'wb')
    impostordict = {}
    n_eid = 0
    for eid in sessdf_eids:
        print(n_eid, len(sessdf_eids), eid)
        n_eid += 1
        impostordict[eid] = get_target_vector_eid(eid, sessdf)
    pickle.dump(impostordict, fimpostor)
    fimpostor.close()
elif 'impostor-session' in SUBDIRECTORY:
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = {}

fit_metadata['submitted_eids'] = sessdf_eids
fit_metadata['impostor_dictionary'] = impostordict
fn = os.path.join(OUTPUT_PATH,SUBDIRECTORY,'DATE_results')
fn = fn + '.parquet'
metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
metadata_fn = metadata_fn.replace('DATE',str(date.today()))
metadata_df.to_pickle(metadata_fn)
print('created metadata file and impostor sessions')

created metadata file and impostor sessions


/home/users/bensonb/mambaforge/envs/iblenv/lib/python3.9/site-packages/one/api.py:1242: UserWarning: Newer cache tables require ONE version 1.10.0 or greater
  warnings.warn(f'Newer cache tables require ONE version {min_version} or greater')


In [ ]:
eid = '15f742e1-1043-45c9-9504-f1e8a53c1744'
one = ONE()  # mode='local'
atlas = AllenAtlas()

estimator = ESTIMATOR #(**ESTIMATOR_KWARGS)

subject = sessdf.xs(eid, level='eid').index[0]
subjeids = sessdf.xs(subject, level='subject').index.unique()
brainreg = dut.BrainRegions()
behavior_data = mut.load_session(eid, one=one)
pLeft_vec = np.array(behavior_data['probabilityLeft'])
try:
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, beh_data=behavior_data,
                              one=one)
except ValueError:
    print('Model not fit.')
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, one=one)

fvecs = [dut.compute_target(feature, subject, subjeids, eid, MODELFIT_PATH,
                          modeltype=MODEL, beh_data=behavior_data,
                          one=one) for feature in CONTROL_FEATURES]

try:
    trialsdf = bbone.load_trials_df(eid, one=one, addtl_types=['firstMovement_times'])
    if len(trialsdf) != len(tvec):
        raise IndexError
except IndexError:
    raise IndexError('Problem in the dimensions of dataframe of session')
trialsdf['react_times'] = trialsdf['firstMovement_times'] - trialsdf['goCue_times']

In [7]:
trialsdf.keys()

Index(['choice', 'probabilityLeft', 'feedbackType', 'feedback_times',
       'contrastLeft', 'contrastRight', 'goCue_times', 'stimOn_times',
       'firstMovement_times', 'trial_start', 'trial_end', 'react_times'],
      dtype='object')

In [12]:
def generate_shifts():
    out = np.arange(-10,10)
    np.random.shuffle(out)
    i = 0
    while i < len(out):
        yield out[i]
        i+=1
        
gs = generate_shifts()
for j in range(30):
    print(next(generate_shifts()))
    

-6
-1
-8
1
-1
7
4
5
7
-10
-5
-3
-5
-9
-1
6
-3
9
0
5
9
8
-7
-1
-3
9
-4
-10
9
1


In [3]:
# submit python file with eid inputs
for eid in sessdf_eids:
    slurm_utils.submit_python_file(py_file, 
                             eid, 
                             ADD_TO_SAVING_PATH = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 8,
                             SLURM_DIRECTORY = SLURM_DIRECTORY,
                             SUBDIRECTORY = SUBDIRECTORY)
    

Submitted batch job 53882486
Submitted batch job 53882487
Submitted batch job 53882488
Submitted batch job 53882489
Submitted batch job 53882490
Submitted batch job 53882491
Submitted batch job 53882492
Submitted batch job 53882493
Submitted batch job 53882494
Submitted batch job 53882495
Submitted batch job 53882496
Submitted batch job 53882497
Submitted batch job 53882498
Submitted batch job 53882499
Submitted batch job 53882500
Submitted batch job 53882501
Submitted batch job 53882502
Submitted batch job 53882503
Submitted batch job 53882504
Submitted batch job 53882505
Submitted batch job 53882506
Submitted batch job 53882507
Submitted batch job 53882508
Submitted batch job 53882509
Submitted batch job 53882510
Submitted batch job 53882511
Submitted batch job 53882512
Submitted batch job 53882513
Submitted batch job 53882514
Submitted batch job 53882515
Submitted batch job 53882516
Submitted batch job 53882517
Submitted batch job 53882518
Submitted batch job 53882519
Submitted batc

Submitted batch job 53882772
Submitted batch job 53882773
Submitted batch job 53882774
Submitted batch job 53882776
Submitted batch job 53882777
Submitted batch job 53882778
Submitted batch job 53882779
Submitted batch job 53882780
Submitted batch job 53882781
Submitted batch job 53882782
Submitted batch job 53882783
Submitted batch job 53882784
Submitted batch job 53882785
Submitted batch job 53882786
Submitted batch job 53882787
Submitted batch job 53882788
Submitted batch job 53882789
Submitted batch job 53882790
Submitted batch job 53882791
Submitted batch job 53882792
Submitted batch job 53882793
Submitted batch job 53882794
Submitted batch job 53882795
Submitted batch job 53882796
Submitted batch job 53882797
Submitted batch job 53882798
Submitted batch job 53882799
Submitted batch job 53882800
Submitted batch job 53882801
Submitted batch job 53882802
Submitted batch job 53882803
Submitted batch job 53882804
Submitted batch job 53882805
Submitted batch job 53882806
Submitted batc

In [2]:

os.system("squeue -u $USER");
slurm_utils.get_decoding_output_files(SLURM_DIRECTORY = SLURM_DIRECTORY,
                                      SUBDIRECTORY = SUBDIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          53889608    normal interact  bensonb  R      12:14      1 sh02-01n56


['/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/KS046/dfbe628d-365b-461c-a07f-8b9911ba83aa/probe01/2022-05-23_AON.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/KS046/dfbe628d-365b-461c-a07f-8b9911ba83aa/probe01/2022-05-23_MOs.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/KS046/dfbe628d-365b-461c-a07f-8b9911ba83aa/probe01/2022-05-23_ORBvl.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/KS046/dfbe628d-365b-461c-a

In [3]:
#SUBDIRECTORY = 'decode_pLeft_task_Logistic_accuracy_control_10_impostor-session_align_goCue_times_timeWin_-0_4_-0_1_testnulltype1'

slurm_utils.gather_save_outputs(SUBDIRECTORY, SLURM_DIRECTORY, OUTPUT_PATH)

successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
failed to open file /home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/KS014/16693458-0801-4d35-a3f1-9115c7e5acfd/probe01/2022-05-23_PAG.pkl
successfully ran relative targ

successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
failed to open file /home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/DY_009/413a6825-2144-4a50-b3fc-cf38ddd6fd1a/probe00/2022-05-23_ENTm

successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successful

successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
failed to open file /home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_pLeft_100_pseudo-session_align_goCue_times_timeWin_0_0_1_all220522/DY_011/741979ce-3f10-443a-8526-2275620c8473/probe00/2022-05-23_AV.pkl
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative target
successfully ran relative targ